# ADNN Colab readiness check
Run this notebook before the full study. This checks the runtime and creates project storage; it does not run experiments or download model weights.

Upload this file through Colab → File → Upload notebook. Select a GPU runtime for the hardware check. Return to CPU for dataset preparation. Never put access tokens in cells or saved output.

Scope: RQ1–RQ3 only. No bonus experiments. Target completion: 23 October 2026; course deadline: 11 November 2026.


In [ ]:
import sys, platform, shutil, subprocess, json, importlib.metadata
from pathlib import Path
report = {"python": sys.version, "platform": platform.platform()}
for package in ["torch", "transformers", "scikit-learn", "numpy", "scienceworld"]:
    try: report[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: report[package] = "not installed"
if shutil.which("nvidia-smi"):
    result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture_output=True, text=True)
    report["gpu"] = result.stdout.strip() or result.stderr.strip()
else:
    report["gpu"] = "No NVIDIA GPU detected. Select a GPU runtime for model work."
report["disk_free_gib"] = round(shutil.disk_usage("/content" if Path("/content").exists() else ".").free / 2**30, 2)
if shutil.which("java"):
    report["java"] = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.strip()
else: report["java"] = "not installed"
print(json.dumps(report, indent=2))


## Persist project files
The next cell requests Google Drive access and creates only a dedicated ADNN_Irreversibility folder. Experiment agents must operate on simulated state, never on your mounted Drive. The experiment harness alone saves results.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/ADNN_Irreversibility")
for folder in ["configs", "data", "splits", "annotations", "activations", "logs", "results", "figures", "paper", "presentation"]:
    (PROJECT / folder).mkdir(parents=True, exist_ok=True)
from datetime import datetime, timezone
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
(PROJECT / "configs" / f"runtime_{stamp}.json").write_text(json.dumps(report, indent=2))
print("Project folder:", PROJECT)


## Next checkpoint
Share the hardware report, without credentials. Before full experiments we must validate model loading, Llama access, ScienceWorld replay, and a timed pilot.

A runtime report is not a tested dependency lock. Freeze exact dependency versions after successful pilot runs. Choose and document model precision once; do not silently combine results from different quantization settings.

Save completed examples in restartable chunks. A Colab disconnect must not force the whole experiment to restart. Keep dataset IDs and configuration hashes with every result. Cache both required token summaries during the same forward pass.

Reference: https://research.google.com/colaboratory/faq.html
